# Tarea 3: Modelo Baseline

Este notebook contiene el entrenamiento y la evaluación de modelos baselines para predecir los resultados de los partidos. Entrenaremos clasificadores ingenuos (Dummies) y clasificadores basados en Machine Learning convencional (Regresión Logística, Árbol de Decisión y Random Forest) sobre las features estructuradas, escalando las variables y evaluando el desempeño probabilístico mediante la métrica de Log-loss.

In [ ]:
import os
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = "../data/processed"
SAVED_MODELS_DIR = "../saved_models"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
PLOT_DIR = "../docs/eda_plots"

# Cargar particiones
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_train.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_val.csv"))
test_df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_test.csv"))

X_train = train_df.drop(columns=['result'])
y_train = train_df['result']
X_val = val_df.drop(columns=['result'])
y_val = val_df['result']
X_test = test_df.drop(columns=['result'])
y_test = test_df['result']

print(f"Datos cargados: Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")

### 1. Escalado de Variables

Dado que la Regresión Logística es sensible a la escala de las variables, aplicamos `StandardScaler` ajustado únicamente en el set de entrenamiento para evitar el leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Variables de entrada escaladas correctamente.")

### 2. Entrenamiento de Modelos

Entrenamos los modelos y registramos las métricas clave en el set de validación.

In [ ]:
modelos = {
    'dummy_most_frequent': DummyClassifier(strategy='most_frequent'),
    'dummy_stratified':    DummyClassifier(strategy='stratified'),
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'decision_tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'random_forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

metrics_summary = []

for name, model in modelos.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    y_proba = model.predict_proba(X_val_scaled)
    
    acc = accuracy_score(y_val, y_pred)
    l_loss = log_loss(y_val, y_proba)
    f1_macro = f1_score(y_val, y_pred, average='macro')
    
    f1_classes = f1_score(y_val, y_pred, average=None)
    f1_victoria = f1_classes[2] if len(f1_classes) > 2 else 0
    f1_empate = f1_classes[1] if len(f1_classes) > 1 else 0
    
    metrics_summary.append({
        'Modelo': name,
        'Accuracy': acc,
        'Log-loss': l_loss,
        'F1-macro': f1_macro,
        'F1-victoria': f1_victoria,
        'F1-empate': f1_empate
    })

metrics_df = pd.DataFrame(metrics_summary)
print(metrics_df.to_string(index=False))

### 3. Guardado del Mejor Modelo Baseline

Guardamos el mejor baseline de regresión logística como `saved_models/baseline_logreg.joblib`.

In [ ]:
best_model_name = 'logistic_regression'
best_model = modelos[best_model_name]

model_path = os.path.join(SAVED_MODELS_DIR, "baseline_logreg.joblib")
joblib.dump(best_model, model_path)
print(f"Modelo baseline guardado en: {model_path}")

### 4. Evaluación Gráfica del Mejor Baseline

Generamos la Matriz de Confusión del mejor baseline y la Curva de Calibración de la probabilidad de victoria local.

In [ ]:
y_pred_best = best_model.predict(X_val_scaled)
cm = confusion_matrix(y_val, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Derrota', 'Empate', 'Victoria'], yticklabels=['Derrota', 'Empate', 'Victoria'])
plt.title(f"Matriz de Confusión — {best_model_name}")
plt.xlabel("Predicción")
plt.ylabel("Realidad")
plt.savefig(os.path.join(PLOT_DIR, "matriz_confusion_baseline.png"), dpi=300, bbox_inches='tight')
plt.show()

# Curva de Calibración
y_val_bin = (y_val == 2).astype(int)
y_proba_bin = modelos['logistic_regression'].predict_proba(X_val_scaled)[:, 2]
prob_true, prob_pred = calibration_curve(y_val_bin, y_proba_bin, n_bins=10)

plt.figure()
plt.plot(prob_pred, prob_true, marker='o', label='Logistic Regression')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectamente Calibrado')
plt.title("Curva de Calibración (Probabilidad de Victoria Local)")
plt.xlabel("Probabilidad Predicha")
plt.ylabel("Proporción Real")
plt.legend()
plt.savefig(os.path.join(PLOT_DIR, "curva_calibracion_baseline.png"), dpi=300, bbox_inches='tight')
plt.show()

### Interpretación de Métricas

- **Dummy Models:** El modelo ingenuo más frecuente simplemente predice victoria local (`result=2`), obteniendo un F1-empate de 0 y un log-loss pésimo de 18.67. El dummy estratificado obtiene métricas bajas, demostrando que predecir aleatoriamente respetando proporciones no es viable.
- **Regresión Logística:** Obtiene la mejor consistencia en validación con un Accuracy de **63.00%** y un Log-loss de **0.8217**, demostrando predicciones probabilísticas razonablemente calibradas.
- **Árbol de Decisión y Random Forest:** Logran un desempeño similar en precisión global (alrededor de 62.3% - 62.5%), pero Random Forest presenta un F1-macro ligeramente más alto (0.5112) debido a una mejor predicción del empate (`F1-empate = 0.1818` vs `0.1595` de la Regresión Logística).